# GenAI-Kickstart
## Lesson: Interacting with an Ollama model

There are several ways to talk to a model served by Ollama. In this notebook we walk through four of them, from the most Ollama-specific to the most generic:

1. **`ollama` SDK** &mdash; Ollama's own Python client. Talks to Ollama's native API and exposes Ollama-specific features directly.
2. **`requests`** &mdash; raw HTTP calls against the OpenAI-compatible endpoint Ollama also exposes. Useful for understanding what is actually sent over the wire.
3. **`openai` client** &mdash; a thin Python wrapper around that same OpenAI-compatible endpoint. Because the protocol is the OpenAI standard, the same client works against Ollama, Gemini, vLLM, and many others just by switching `base_url` and `api_key`.
4. **Streaming** &mdash; receiving the answer token-by-token instead of waiting for the full response.

### Set up the environment

In [1]:
import requests
from dotenv import load_dotenv, find_dotenv
from ollama import Client as OllamaClient
from openai import OpenAI

load_dotenv(find_dotenv())

True

In [2]:
# The model must already be pulled in your local Ollama (e.g. `ollama pull deepseek-r1:1.5b`)
OLLAMA_MODEL = "deepseek-r1:1.5b"

# Native Ollama host (used by the ollama SDK)
OLLAMA_HOST = "http://localhost:11434"

# OpenAI-compatible endpoint exposed by Ollama (used by requests and the OpenAI client)
OLLAMA_BASE_URL = f"{OLLAMA_HOST}/v1"

# Ollama ignores the API key, but the OpenAI client requires *some* value
OLLAMA_API_KEY = "ollama"

# A single message we will reuse across all four techniques so the calls are easy to compare
messages = [
    {"role": "user", "content": "Why is the sky blue? Answer in one short sentence."},
]

### `ollama` SDK

The `ollama` Python package is the official SDK and talks to Ollama's native API at `/api/chat`. Compared to the other approaches in this notebook, it is the most Ollama-specific &mdash; the request and response objects mirror Ollama's own model, and Ollama-only features (e.g. model management, embeddings, structured outputs via the `format` parameter) are first-class here.

In [3]:
ollama_client = OllamaClient(host=OLLAMA_HOST)

response = ollama_client.chat(
    model=OLLAMA_MODEL,
    messages=messages,
)

print(response.message.content)

KeyboardInterrupt: 

### `requests` library

The most explicit option: build the HTTP request ourselves. The endpoint, headers, and body shape are exactly what the OpenAI spec defines &mdash; the `openai` client will do the same thing under the hood.

In [ ]:
response = requests.post(
    f"{OLLAMA_BASE_URL}/chat/completions",
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
    json={
        "model": OLLAMA_MODEL,
        "messages": messages,
    },
)
response.raise_for_status()
payload = response.json()

# Inspect the raw payload once so the OpenAI-compatible shape is visible
payload

{'id': 'chatcmpl-698',
 'object': 'chat.completion',
 'created': 1777145087,
 'model': 'deepseek-r1:1.5b',
 'system_fingerprint': 'fp_ollama',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "The sky appears blue because of a combination of factors, including Earth's atmosphere absorbing colored light (blue and violet) more efficiently than shorter wavelength colors like red and yellow when they reach the observer's eye. Additionally, atmospheric pollution and particulate matter can further degrade this effect, contributing to the overall blue coloration we observe."},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 15, 'completion_tokens': 70, 'total_tokens': 85}}

In [ ]:
# Pull the assistant's reply out of the response payload
print(payload["choices"][0]["message"]["content"])

The sky appears blue because of a combination of factors, including Earth's atmosphere absorbing colored light (blue and violet) more efficiently than shorter wavelength colors like red and yellow when they reach the observer's eye. Additionally, atmospheric pollution and particulate matter can further degrade this effect, contributing to the overall blue coloration we observe.


### `openai` client

Same endpoint, same JSON shape &mdash; but we let the `openai` package handle the HTTP plumbing and give us typed objects back. Because Ollama speaks the OpenAI protocol, switching providers (e.g. to [Gemini's OpenAI-compatible endpoint](https://ai.google.dev/gemini-api/docs/openai)) is just a matter of changing `base_url` and `api_key`.

In [ ]:
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
)

chat_completion = client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=messages,
)

print(chat_completion.choices[0].message.content)

The sky appears blue due to Rayleigh scattering, where shorter blue wavelengths are scattered more by atmospheric air molecules, leaving red and orange hues scattered away.


### Streaming

With `stream=True` the model emits the answer token-by-token. Each chunk is a partial completion; concatenating the `delta.content` fields rebuilds the final answer. This is what powers the typewriter effect in chat UIs.

In [ ]:
chat_stream = client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=messages,
    stream=True,
)

for chunk in chat_stream:
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)


The blue sky is primarily due to solar diversity as seen in various cloud forms and the scattering of sunlight by particles in the air, such as water droplets and dust, which cause some regions to appear darker with less light scattering. Additionally, wind movements contribute through atmospheric shifts, affecting how light scatters across clouds.